# Feature Transformers for Cross-Domain Generalization

This notebook demonstrates the `StructuralFeatures` and `SpectralFeatures`
transformers, which compute fixed-dimensional node features from graph
topology alone. This enables training models that can generalize across
graphs from different domains (brain connectivity, document similarity,
social networks, etc.).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nocd import NOCD, StructuralFeatures, SpectralFeatures
from nocd.data import load_dataset, prepare_features
from nocd.metrics import overlapping_nmi, evaluate_unsupervised

In [ ]:
graph = load_dataset('data/mag_cs.npz')
A, X, Z_gt = graph['A'], graph['X'], graph['Z']
N, K = Z_gt.shape
print(f'Graph: {N} nodes, {A.nnz} edges, {K} communities')

## StructuralFeatures transformer

Computes 9 topology-derived features per node:
normalized degree, log-degree, clustering coefficient, square clustering,
average neighbor degree, PageRank, HITS hub/authority scores, and core number.

These features have the same dimensionality regardless of graph size or domain.

In [ ]:
sf = StructuralFeatures()
X_struct = sf.fit_transform(A)
print(f'Structural features: {X_struct.shape}')
print(f'Feature names: degree, log_degree, clustering, sq_clustering, '
      f'avg_nbr_degree, pagerank, hub, authority, core_number')
print(f'\nSample (node 0): {X_struct[0]}')

In [ ]:
# Visualize feature distributions
names = ['degree', 'log_deg', 'clust', 'sq_clust',
         'avg_nbr_deg', 'pagerank', 'hub', 'auth', 'core']

fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for i, (ax, name) in enumerate(zip(axes.flat, names)):
    ax.hist(X_struct[:, i], bins=50, alpha=0.7)
    ax.set_title(name)
plt.suptitle('Structural Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## SpectralFeatures transformer

Computes the top-k smallest non-trivial eigenvectors of the normalized
graph Laplacian. These capture global graph structure as positional encodings.

In [ ]:
spf = SpectralFeatures(n_components=16)
X_spec = spf.fit_transform(A)
print(f'Spectral features: {X_spec.shape}')

In [ ]:
# Visualize first two spectral components as a 2D embedding
z_gt_label = np.argmax(Z_gt, axis=1)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_spec[:, 0], X_spec[:, 1], c=z_gt_label,
                     cmap='tab20', s=1, alpha=0.5)
ax.set_xlabel('Spectral component 1')
ax.set_ylabel('Spectral component 2')
ax.set_title('Spectral embedding colored by ground-truth community')
plt.colorbar(scatter, label='Community')
plt.tight_layout()
plt.show()

## Training with structural features

Because structural features are 9-dimensional regardless of graph size,
a model trained here can be applied to any other graph.

In [ ]:
model_struct = NOCD(
    num_communities=K,
    model_type='gcn',
    hidden_dims=(64, 32),
    feature_type='structural',
    batch_norm=True,
    max_epochs=500,
)
model_struct.fit(A, y=Z_gt)

In [ ]:
Z_struct = model_struct.predict(A)
nmi_struct = overlapping_nmi(Z_struct, Z_gt)
print(f'Structural features NMI: {nmi_struct:.4f}')
print('Unsupervised metrics:', evaluate_unsupervised(Z_struct, A))

## Training with spectral features

In [ ]:
model_spec = NOCD(
    num_communities=K,
    model_type='gcn',
    hidden_dims=(64, 32),
    feature_type='spectral',
    n_components=32,
    batch_norm=True,
    max_epochs=500,
)
model_spec.fit(A, y=Z_gt)

In [ ]:
Z_spec = model_spec.predict(A)
nmi_spec = overlapping_nmi(Z_spec, Z_gt)
print(f'Spectral features NMI: {nmi_spec:.4f}')
print('Unsupervised metrics:', evaluate_unsupervised(Z_spec, A))

## Comparison

Compare the different feature types on the same dataset.

In [ ]:
# Also train with original node features for comparison
model_X = NOCD(
    num_communities=K,
    model_type='gcn',
    hidden_dims=(128,),
    feature_type='X',
    batch_norm=True,
    max_epochs=500,
)
model_X.fit(A, X, y=Z_gt, verbose=False)
nmi_X = overlapping_nmi(model_X.predict(A, X), Z_gt)

print(f'Node attributes (X):    NMI = {nmi_X:.4f}')
print(f'Structural features:    NMI = {nmi_struct:.4f}')
print(f'Spectral features:      NMI = {nmi_spec:.4f}')
print()
print('Note: X features are domain-specific and achieve the best NMI.')
print('Structural/spectral features are domain-agnostic and transferable.')

## Cross-graph transfer

Demonstrate that a model trained with structural features on one graph
can be applied to a different graph (different size, different domain).

In [ ]:
# Load a different dataset
graph2 = load_dataset('data/mag_eng.npz')
A2, Z_gt2 = graph2['A'], graph2['Z']
N2, K2 = Z_gt2.shape
print(f'Second graph: {N2} nodes, {A2.nnz} edges, {K2} communities')

# Use the model trained on mag_cs to predict on mag_eng
# (This works because structural features have fixed dimensionality)
Z_transfer = model_struct.predict(A2)
unsup_transfer = evaluate_unsupervised(Z_transfer, A2)
print(f'\nTransferred model unsupervised metrics on mag_eng:')
for k, v in unsup_transfer.items():
    print(f'  {k}: {v:.4f}')